# Chapter 5: Function-Calling — teach the base to use tools

Every chapter has hit the same wall: a ~50M model **cannot be a knowledge store**. Ch.3's chat model said the capital of France was "Germany"; Ch.4's broader data made it *fluent about* capitals but still got the fact wrong. You cannot memorize your way out of 50M parameters.

**This chapter stops trying.** Instead of forcing the model to *know* facts, we teach it to **call a tool** that knows them — a search function, a calculator, a weather API. The model's job shrinks to something a tiny model *can* do well: read a question, decide which tool to call, and emit a correctly-formatted call. A harness runs the tool and feeds the answer back. That is the core of an **agent**.

### The agentic loop (the whole idea, one picture)

```
User: "What is 18 percent of 245?"
   │
   ▼  model emits a TOOL CALL (structured JSON):
   {"name": "calculator", "arguments": {"expression": "245 * 0.18"}}
   │
   ▼  the harness PARSES the JSON and EXECUTES the real function -> 44.1
   │
   ▼  the result is the grounded answer (no hallucinated arithmetic)
```

### Why this works at 50M — tools live in the *prompt*

The trick that makes function-calling learnable for a tiny model: the **list of available tools is given in the prompt**, as JSON. So the model never has to *memorize* tools — it reads what's on offer this turn and learns the transferable skill *"map (question + available tools) → the right call"*. That skill generalizes to tools it never saw in training, because they're always supplied in context. This is exactly how real function-calling models work, and it's why a small model has a fighting chance here.

### Real data: xLAM-60k

We use [`Salesforce/xlam-function-calling-60k`](https://huggingface.co/datasets/Salesforce/xlam-function-calling-60k) — 60k real `(query, tools, answer-calls)` examples across thousands of tools. Each row already has the three things we need:
- `query` — the user's request,
- `tools` — the JSON list of tools available *for that example*,
- `answers` — the JSON list of calls the model should produce.

> **Single-turn, by design.** xLAM teaches the *hard* new skill — producing the correct structured call. It does **not** include the model *narrating* the tool's result in prose (that needs multi-turn data like Glaive-function-calling-v2 — noted as the scale-up at the end). So our agentic loop ends at "execute and return the grounded result", which is already the fix for the France problem.

### What's new vs Chapter 3b

| Concept | 3b (chat SFT) | 5 (tool SFT) |
|---|---|---|
| the prompt | instruction | instruction **+ a JSON list of available tools** |
| the response (what we train on) | free text | a **structured JSON tool call** |
| after generation | done | **parse the JSON → execute a real function → return the result** |
| the win | answers in-format | answers are **grounded** (the tool knows what the model can't) |

Everything else is 3b machinery: the same **loss masking** (train only on the response — here, the call), the same SFT loop. We SFT the **Ch.4 modern base** (`modern.pt`). Same concept→TODO→check pattern.

## 0. Setup — load the Ch.4 modern base

We reuse the Llama-style architecture from Ch.4 (given again so this notebook stands alone) and load `data/modern.pt`. If you haven't run Ch.4, do that first.

**A free upgrade we exploit here:** function-calling prompts are *long* (a JSON tool list + the query), often past the base's trained `block_size=256`. Because Ch.4 dropped the learned position table for **RoPE**, there is **no positional parameter to resize** — we can rebuild the exact same weights at `block_size=512` and RoPE just keeps rotating. We're literally cashing in the "length-extrapolation, zero-param positions" benefit from Ch.4. (The base only *trained* at 256, so this is mild extrapolation — fine for SFT.)

In [ ]:
import math, os, json, time, ast, re
from dataclasses import dataclass, replace

import torch
import torch.nn.functional as F
from torch import nn
import tiktoken

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
DATA_DIR = "data"
enc = tiktoken.get_encoding("gpt2")
EOT = enc.eot_token
VOCAB_SIZE = enc.n_vocab
print("device:", device)

In [ ]:
# --- Ch.4 Llama-style architecture, reproduced verbatim (GIVEN -- you built this in Ch.4) ---
@dataclass
class Config:
    vocab_size: int = VOCAB_SIZE
    d_model: int = 512
    n_heads: int = 8
    n_kv_heads: int = 2
    d_ff: int = 1408
    n_layers: int = 8
    block_size: int = 256
    rope_theta: float = 10000.0
    dropout: float = 0.0
    def __post_init__(self):
        assert self.d_model % self.n_heads == 0
        assert self.n_heads % self.n_kv_heads == 0


def precompute_rope(d_k, max_pos, theta=10000.0, device="cpu"):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_k, 2, device=device) / d_k))
    t = torch.arange(max_pos, device=device)
    freqs = torch.outer(t, inv_freq)
    emb = torch.cat([freqs, freqs], dim=-1)
    return emb.cos(), emb.sin()

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(q, k, cos, sin):
    return q * cos + rotate_half(q) * sin, k * cos + rotate_half(k) * sin

def repeat_kv(x, n_rep):
    if n_rep == 1:
        return x
    b, n_kv, t, d = x.shape
    return x[:, :, None, :, :].expand(b, n_kv, n_rep, t, d).reshape(b, n_kv * n_rep, t, d)


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x / rms) * self.weight

class GroupedQueryAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads = cfg.n_heads; self.n_kv = cfg.n_kv_heads
        self.n_rep = cfg.n_heads // cfg.n_kv_heads; self.d_k = cfg.d_model // cfg.n_heads
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_heads * self.d_k, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.d_k, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.d_k, bias=False)
        self.o_proj = nn.Linear(cfg.n_heads * self.d_k, cfg.d_model, bias=False)
        self.dropout = cfg.dropout
    def forward(self, x, cos, sin, past_kv=None, use_cache=False, is_causal=True):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv, self.d_k).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv, self.d_k).transpose(1, 2)
        q, k = apply_rope(q, k, cos, sin)
        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2); v = torch.cat([past_kv[1], v], dim=2)
        present = (k, v) if use_cache else None
        k = repeat_kv(k, self.n_rep); v = repeat_kv(v, self.n_rep)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=is_causal,
            dropout_p=self.dropout if self.training else 0.0)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1)), present

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class LlamaBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model); self.attn = GroupedQueryAttention(cfg)
        self.mlp_norm = RMSNorm(cfg.d_model); self.mlp = SwiGLU(cfg.d_model, cfg.d_ff)
    def forward(self, x, cos, sin, past_kv=None, use_cache=False, is_causal=True):
        a, present = self.attn(self.attn_norm(x), cos, sin, past_kv, use_cache, is_causal)
        x = x + a
        return x + self.mlp(self.mlp_norm(x)), present

class LlamaGPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg; self.block_size = cfg.block_size; self.n_layers = cfg.n_layers
        self.wte = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([LlamaBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm_f = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight
        d_k = cfg.d_model // cfg.n_heads
        cos, sin = precompute_rope(d_k, cfg.block_size, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        x = self.drop(self.wte(idx))
        for block in self.blocks:
            x, _ = block(x, cos, sin, is_causal=True)
        logits = self.lm_head(self.norm_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss


@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=0.4, top_k=40):
    """KV-cached generation (Ch.4). Lower default temperature -- structured output wants to be precise."""
    model.eval()
    caches = [None] * len(model.blocks)
    def step(tokens, start, is_causal):
        T = tokens.shape[1]
        cos, sin = model.rope_cos[start:start + T], model.rope_sin[start:start + T]
        x = model.wte(tokens)
        for i, b in enumerate(model.blocks):
            x, caches[i] = b(x, cos, sin, past_kv=caches[i], use_cache=True, is_causal=is_causal)
        return model.lm_head(model.norm_f(x))[:, -1, :]
    pos = idx.shape[1]
    logits = step(idx, 0, True)
    for _ in range(max_new_tokens):
        logits = logits / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, top_k); logits[logits < v[:, [-1]]] = float("-inf")
        nxt = torch.multinomial(F.softmax(logits, dim=-1), 1)
        idx = torch.cat([idx, nxt], dim=1)
        if nxt.item() == EOT:
            break
        logits = step(nxt, pos, False); pos += 1
    model.train()
    return idx

In [ ]:
# load the modern base, REBUILT at block_size=512 (RoPE extrapolation -- no positional params to resize)
SFT_BLOCK = 512
ckpt = torch.load(os.path.join(DATA_DIR, "modern.pt"), map_location=device, weights_only=False)
sft_cfg = replace(ckpt["cfg"], block_size=SFT_BLOCK)        # only the RoPE buffer length changes; weights are identical
model = LlamaGPT(sft_cfg).to(device)
missing, unexpected = model.load_state_dict(ckpt["model"], strict=True)
print(f"loaded modern base: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params, "
      f"pretrained {ckpt.get('step','?')} steps, now serving block_size={model.block_size}")

## 1. The dataset — xLAM-60k

Each row is exactly the three pieces we need. Let's look at one. `tools` and `answers` arrive as **JSON strings** (so we `json.loads` them to inspect, but for the prompt we keep them as compact JSON text).

In [ ]:
from datasets import load_dataset

raw = load_dataset("Salesforce/xlam-function-calling-60k", split="train")
print(raw)

_ex = raw[0]
print("\nquery  :", _ex["query"])
print("tools  :", _ex["tools"][:400], "...")
print("answers:", _ex["answers"])

## 2. The function-calling template

We extend 3b's Alpaca-style text template with a **tools block**. No new special tokens (same ponytail reasoning as 3b — the BPE tokenizer already handles `### Tools:` and JSON punctuation):

```
### Tools:
[{"name": "...", "description": "...", "parameters": {...}}, ...]

### Instruction:
{query}

### Response:
{answer_calls_json}<|endoftext|>
```

- **prompt** = everything up to and including `### Response:\n` (the tools + the user query). This is *context* — we will mask it.
- **response** = the `answers` JSON (the list of calls), followed by EOT. This is what we **train on**.

We keep `tools` and `answers` as their **compact JSON strings** so the model learns to emit exactly parseable JSON.

| variable | meaning |
|---|---|
| `ex` | one xLAM row: `{query, tools (json str), answers (json str)}` |
| `prompt_str` | tools block + instruction, ending at `### Response:\n` |
| `response_str` | the `answers` JSON string (the call(s) to emit) |

In [ ]:
def format_example(ex):
    """Return (prompt_str, response_str) for one xLAM row.

    Fields: ex["query"] (str), ex["tools"] (JSON str list), ex["answers"] (JSON str list of calls).
    Build prompt_str EXACTLY up to and including "### Response:\n".
    Keep tools/answers as compact JSON text (re-dump so spacing is consistent and parseable).

    Template:
      "### Tools:\n{tools_json}\n\n### Instruction:\n{query}\n\n### Response:\n"
      response_str = {answers_json}
    """
    # TODO:
    # 1. tools_json   = json.dumps(json.loads(ex["tools"]),   separators=(",", ":"))   # compact, normalized
    #    answers_json = json.dumps(json.loads(ex["answers"]), separators=(",", ":"))
    # 2. prompt_str = f"### Tools:\n{tools_json}\n\n### Instruction:\n{ex['query']}\n\n### Response:\n"
    # 3. response_str = answers_json
    # 4. return prompt_str, response_str
    raise NotImplementedError

In [ ]:
# check -- prompt ends at the response marker, tools+query are in the prompt, response is valid JSON calls
_ex = {"query": "What is 12 times 7?",
       "tools": '[{"name":"calculator","description":"do math","parameters":{"expression":{"type":"string"}}}]',
       "answers": '[{"name":"calculator","arguments":{"expression":"12*7"}}]'}
_p, _r = format_example(_ex)
assert _p.endswith("### Response:\n"), "prompt must end exactly at the response marker"
assert "### Tools:" in _p and "calculator" in _p, "tools block must be in the prompt"
assert "What is 12 times 7?" in _p, "query must be in the prompt"
_calls = json.loads(_r)                                   # response must be valid JSON
assert _calls[0]["name"] == "calculator" and "arguments" in _calls[0], "response must be the call list"
print("ok: template splits cleanly; response is parseable JSON\n"); print(_p + _r)

## 3. Loss masking — same idea as 3b, now masking everything but the call

Identical mechanics to 3b: tokenize prompt and response separately, concatenate, and set `labels = -100` over the prompt so cross-entropy trains **only on the tool call**. We do **not** want to teach the model to generate tool definitions or user queries — only to produce the right call.

The only wrinkle vs 3b is length: tool lists are long, so we use `block_size=512` and drop examples that still don't fit.

In [ ]:
def encode_example(ex, block_size):
    """Return (ids, labels) or None if too long. Mask the prompt with -100, train on the call+EOT.

    Steps (identical to 3b):
      1. prompt_str, response_str = format_example(ex)
      2. prompt_ids   = enc.encode_ordinary(prompt_str)
         response_ids = enc.encode_ordinary(response_str) + [EOT]
      3. ids    = prompt_ids + response_ids
      4. labels = [-100]*len(prompt_ids) + response_ids
      5. if len(ids) > block_size: return None
      6. return ids, labels
    """
    # TODO:
    raise NotImplementedError

In [ ]:
# check -- mask covers exactly the prompt; the trainable tokens are the call + EOT
_ids, _labels = encode_example(_ex, block_size=512)
assert len(_ids) == len(_labels), "ids and labels must align"
for a, b in zip(_ids, _labels):
    assert b == -100 or b == a, "every unmasked label must equal its token id"
assert _labels[-1] == EOT, "last trained label must be EOT"
assert _labels[0] == -100, "the prompt must be masked"
n_train = sum(1 for b in _labels if b != -100)
print(f"ok: {len(_ids)} tokens, {n_train} trainable (the JSON call + EOT)")

In [ ]:
from torch.utils.data import DataLoader

# pre-encode the dataset; drop examples that don't fit in block_size (long tool lists)
encoded = []
for ex in raw:
    out = encode_example(ex, SFT_BLOCK)
    if out is not None:
        encoded.append(out)
print(f"kept {len(encoded)}/{len(raw)} examples after the {SFT_BLOCK}-token length filter")


def collate(batch):
    """Right-pad to the batch max, next-token shift (identical to 3b)."""
    L = max(len(ids) for ids, _ in batch)
    ids_pad = torch.full((len(batch), L), EOT, dtype=torch.long)
    lab_pad = torch.full((len(batch), L), -100, dtype=torch.long)
    for i, (ids, labels) in enumerate(batch):
        ids_pad[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)
        lab_pad[i, :len(labels)] = torch.tensor(labels, dtype=torch.long)
    return ids_pad[:, :-1], lab_pad[:, 1:]

## 4. SFT (given — the 3b instrumented loop)

Same gentle fine-tuning as 3b: low LR, a couple epochs, bf16 AMP, grad-clip, a tqdm bar, and periodic generation so you can watch the model learn to emit calls. We fine-tune the modern base in place and save `data/toolcaller.pt`.

In [ ]:
from tqdm.auto import tqdm

# probe prompts use OUR toolset (defined in Phase 5) so we can eyeball progress mid-training
PROBE = [
    ('What is 18% of 245?', '[{"name":"calculator","description":"evaluate a math expression","parameters":{"expression":{"type":"string"}}}]'),
    ('What is the capital of France?', '[{"name":"search","description":"look up a fact","parameters":{"query":{"type":"string"}}}]'),
]

def tool_prompt(query, tools_json):
    return f"### Tools:\n{tools_json}\n\n### Instruction:\n{query}\n\n### Response:\n"

@torch.no_grad()
def raw_generate(model, prompt_str, max_new_tokens=64):
    ids = torch.tensor([enc.encode_ordinary(prompt_str)], device=device)
    out = generate(model, ids, max_new_tokens, temperature=0.3, top_k=40)
    return enc.decode(out[0, ids.shape[1]:].tolist()).split("<|endoftext|>")[0].strip()

def sft_train(model, loader, epochs=2, lr=2e-5, grad_clip=1.0, log_interval=100, sample_interval=400):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.0)
    hist = {"step": [], "loss": [], "grad_norm": [], "vram_gb": []}
    model.train()
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    total = epochs * len(loader); pbar = tqdm(total=total, desc="sft-tools", dynamic_ncols=True); g = 0
    for epoch in range(epochs):
        for x, y in loader:
            t0 = time.time(); x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device, dtype=torch.bfloat16):
                _, loss = model(x, y)
            loss.backward()
            gn = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()
            if device == "cuda": torch.cuda.synchronize()
            vram = torch.cuda.max_memory_allocated()/1e9 if device == "cuda" else 0.0
            hist["step"].append(g); hist["loss"].append(loss.item()); hist["grad_norm"].append(gn.item()); hist["vram_gb"].append(vram)
            pbar.update(1); pbar.set_postfix(ep=epoch, loss=f"{loss.item():.3f}", gnorm=f"{gn.item():.2f}", vram=f"{vram:.1f}G")
            if g % log_interval == 0:
                tqdm.write(f"epoch {epoch} step {g:5d} | loss {loss.item():.3f} | gnorm {gn.item():.2f} | vram {vram:.2f}GB")
            if sample_interval and g > 0 and g % sample_interval == 0:
                for q, tj in PROBE:
                    tqdm.write(f"   [{g}] {q!r} -> {raw_generate(model, tool_prompt(q, tj))!r}")
            g += 1
    pbar.close()
    return hist

In [ ]:
BATCH_SIZE = 16
EPOCHS = 2
LR = 2e-5

loader = DataLoader(encoded, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
hist = sft_train(model, loader, epochs=EPOCHS, lr=LR)
torch.save({"model": model.state_dict(), "cfg": model.cfg}, os.path.join(DATA_DIR, "toolcaller.pt"))
print("saved data/toolcaller.pt")

## 5. The tools + parse-and-execute

A tool is just a **real Python function** plus a **JSON schema** describing it (the schema is what goes in the prompt; the function is what the harness runs). We define a small, honest toolset:

- `search(query)` — a tiny built-in knowledge lookup (stands in for a real search API),
- `calculator(expression)` — safe arithmetic via `ast` (never `eval`),
- `get_weather(location)` — a canned stub.

`parse_and_execute` takes whatever the model emitted, pulls out the JSON call(s), and dispatches each to the matching function. The model output may have trailing junk after the JSON, so we parse defensively (find the first balanced JSON array/object).

In [ ]:
# ---- real tool implementations (given) ----
_FACTS = {
    "capital of france": "Paris", "capital of japan": "Tokyo", "capital of italy": "Rome",
    "capital of germany": "Berlin", "tallest mountain": "Mount Everest (8849 m)",
    "speed of light": "299,792,458 m/s", "author of hamlet": "William Shakespeare",
}
def search(query):
    q = query.lower().strip().rstrip("?")
    for k, v in _FACTS.items():
        if k in q:
            return v
    return "no result found"

def calculator(expression):
    """Safe arithmetic: parse with ast, allow only numbers and +-*/() ** %."""
    node = ast.parse(expression, mode="eval").body
    def ev(n):
        if isinstance(n, ast.Constant): return n.value
        if isinstance(n, ast.BinOp):
            a, b = ev(n.left), ev(n.right)
            return {ast.Add: a+b, ast.Sub: a-b, ast.Mult: a*b, ast.Div: a/b,
                    ast.Pow: a**b, ast.Mod: a%b}[type(n.op)]
        if isinstance(n, ast.UnaryOp) and isinstance(n.op, ast.USub): return -ev(n.operand)
        raise ValueError("unsupported expression")
    return ev(node)

def get_weather(location):
    return f"{location}: 22C, sunny (stub)"

TOOLS = {"search": search, "calculator": calculator, "get_weather": get_weather}

# the schemas we put in the PROMPT (xLAM-style). The model reads these to know what it may call.
TOOL_SCHEMAS = json.dumps([
    {"name": "search", "description": "look up a fact or general knowledge", "parameters": {"query": {"type": "string"}}},
    {"name": "calculator", "description": "evaluate a math expression", "parameters": {"expression": {"type": "string"}}},
    {"name": "get_weather", "description": "current weather for a location", "parameters": {"location": {"type": "string"}}},
], separators=(",", ":"))

In [ ]:
def _first_json(text):
    """Extract the first balanced JSON array/object substring from `text` (model output may trail junk)."""
    start = None
    for i, ch in enumerate(text):
        if ch in "[{":
            start = i; open_ch, close_ch = ch, "]" if ch == "[" else "}"; break
    if start is None:
        return None
    depth = 0
    for j in range(start, len(text)):
        if text[j] in "[{": depth += 1
        elif text[j] in "]}": depth -= 1
        if depth == 0:
            return text[start:j + 1]
    return None


def parse_and_execute(model_output, tools=TOOLS):
    """Parse the model's JSON call(s) and run the real function(s). Returns a list of
    {name, arguments, result} (result is an error string if the call is malformed/unknown)."""
    # TODO:
    # 1. blob = _first_json(model_output);  if None -> return [{"error": "no JSON call found"}]
    # 2. calls = json.loads(blob);  if it's a dict (single call) wrap it: calls = [calls]
    # 3. for each call: name = call["name"]; args = call.get("arguments", {})
    #       if name not in tools -> result = f"unknown tool: {name}"
    #       else try: result = tools[name](**args)   except Exception as e: result = f"error: {e}"
    #       collect {"name": name, "arguments": args, "result": result}
    # 4. return the list
    raise NotImplementedError

In [ ]:
# check -- parse + execute on hand-written model outputs (don't need a trained model for this)
_o1 = '[{"name":"calculator","arguments":{"expression":"245*0.18"}}]'
_r1 = parse_and_execute(_o1)
assert abs(_r1[0]["result"] - 44.1) < 1e-6, "calculator call should execute"

_o2 = '[{"name":"search","arguments":{"query":"capital of France"}}] then some trailing model junk'
_r2 = parse_and_execute(_o2)
assert _r2[0]["result"] == "Paris", "search call should ground the fact the model never memorized"

_o3 = '{"name":"get_weather","arguments":{"location":"Berlin"}}'      # single dict, not a list
assert "Berlin" in parse_and_execute(_o3)[0]["result"], "must accept a single-call dict too"

_o4 = 'I think the answer is 5'                                        # no JSON at all
assert "error" in parse_and_execute(_o4)[0], "non-JSON output must return an error, not crash"
print("ok: parse_and_execute runs real tools and fails gracefully")

## 6. The agentic loop

Now the payoff. `chat_with_tools`:
1. builds the prompt with **our** `TOOL_SCHEMAS` + the user query,
2. generates the model's tool call,
3. **parses and executes** it,
4. returns the call + the grounded result.

This is a single tool step (xLAM is single-turn). The result *is* the grounded answer — which is the whole point: the model that couldn't store "Paris" can now *call a tool* that returns it. (Narrating the result back in fluent prose — "The capital of France is Paris." — is the multi-turn skill that Glaive-style data adds; see the closing notes.)

In [ ]:
def chat_with_tools(model, query, tools_json=TOOL_SCHEMAS, verbose=True):
    """Run one agentic step: prompt -> tool call -> parse -> execute -> grounded result."""
    # TODO:
    # 1. prompt = tool_prompt(query, tools_json)          # from Phase 4 (### Tools / ### Instruction / ### Response)
    # 2. raw = raw_generate(model, prompt, max_new_tokens=64)   # the model's emitted call (string)
    # 3. results = parse_and_execute(raw)
    # 4. if verbose: print the query, the raw call, and the executed result(s)
    # 5. return results
    raise NotImplementedError

In [ ]:
# ---- demo: ask things a 50M model could never memorize; watch it call tools instead ----
for q in [
    "What is the capital of France?",
    "What is 18 percent of 245?",
    "What's the weather in Berlin?",
    "Who wrote Hamlet?",
]:
    chat_with_tools(model, q)
    print("-" * 70)

## Done — a (tiny) tool-using agent

You SFT'd the modern base on **real** function-calling data (xLAM-60k) and wrapped it in a parse-and-execute loop. The model no longer has to *know* the capital of France — it learned to **call a tool** that does, with the tools supplied in the prompt so the skill generalizes.

### What worked / what didn't (be honest with the output)
- **Format:** at 50M the big win is that it emits *parseable, schema-matching JSON calls* for tools it's shown — that's the new, hard skill, and it's learnable precisely because tools live in the prompt.
- **Reliability:** expect it to sometimes pick the wrong tool, mangle an argument, or trail junk after the JSON (that's why `parse_and_execute` is defensive). A bigger base (Ch.4's scale-up) makes the calls far more reliable.

### Scale-up knobs (concepts unchanged)
- **Answer narration (the multi-turn loop):** swap/augment with `glaiveai/glaive-function-calling-v2`, whose conversations include the **tool result fed back** and the assistant's **final prose answer**. Then `chat_with_tools` gains a step 5: feed the result into the context and generate the natural-language reply.
- **More tools / multi-step:** the same loop runs N times (call → execute → observe → call again) for multi-hop agents (ReAct-style).
- **Bigger base:** more pretraining (Ch.4) is still the highest-leverage fix — a stronger base makes a far more reliable tool-caller.

### Where this lands the project
This is the **agentic** end goal from the handoff, in miniature: pretrain (Ch.3/4) → SFT a behavior (Ch.3b chat, Ch.5 tools). The remaining end goal is **multilingual**, which reuses Ch.4's mixture loader with language slices + a multilingual tokenizer — same machinery, different data.